# Evaluation Rubric

Deterministic scoring criteria for evaluating model-generated product descriptions.
Each criterion is rated as good, ok, or bad using explicit measurable rules.


## Criterion Definitions

### 1. Fluency (Natural, easy-to-read sentences)

good:
Sentences read naturally with smooth flow and no awkward phrasing.

ok:
Minor awkward phrasing but overall understandable and readable.

bad:
Multiple unnatural or difficult-to-follow sentences that disrupt readability.


### 2. Grammar (Correct spelling & punctuation)

good:
No spelling or punctuation errors.

ok:
1 minor grammar/spelling/punctuation error.

bad:
2 or more grammar/spelling/punctuation errors.


### 3. Tone (Friendly, credible sales voice)

good:
Consistently friendly, engaging, and credible sales tone throughout.

ok:
Mostly appropriate tone with minor inconsistency OR slightly neutral wording.

bad:
Tone inappropriate, overly formal, robotic, or not aligned with sales voice.


### 4. Length (50–90 words)

good:
50–90 words

ok:
40–49 words OR 91–110 words

bad:
Less than 40 words OR more than 110 words


### 5. Grounding (The response contains information included only in the context. The response does not reference any outside information.)

good:
All claims strictly supported by provided information.

ok:
One minor unsupported detail added.

bad:
Multiple unsupported claims OR contradicts provided information.


### 6. Latency (Average time per call)

good:
Response time within acceptable production threshold (e.g., ≤2 seconds).

ok:
Response time slightly above threshold (2–4 seconds).

bad:
Response time exceeds acceptable limit (>4 seconds).


### 7. Cost (Average price per call per 1K tokens)

good:
Within target budget threshold.

ok:
Slightly above target budget (≤25% over).

bad:
More than 25% above target budget threshold.

## Pass / Fail Definition

### Cumulative pass rule

A response passes if:

- At least 4 criteria are rated good
- No more than 1 criterion is rated bad


### Automatic failure rules (go / no-go)

A response automatically fails if:

- Grounding is not good
OR
- Grammar is bad

# LLM Product description generation

In [3]:
system_prompt = """
You are a professional ecommerce product description writer.

Write concise, engaging product descriptions using:

- product name
- key features
- material
- warranty

Follow these rules:

1. Length: 40–60 words
2. Tone: professional and persuasive
3. Highlight 2–3 key features
4. Mention material if relevant
5. Mention warranty at the end
6. Do NOT invent information
"""

In [4]:
%pip install pandas openpyxl


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd

df = pd.read_excel("Assignment_01_product_dataset.xlsx")

In [10]:
def format_product(row):
    return f"""
Product name: {row['product_name']}
Attributes: {row['Product_attribute_list']}
Material: {row['material']}
Warranty: {row['warranty']}
"""



In [11]:
format_product(df.iloc[0])

'\nProduct name: Apple iPhone 15 Pro\nAttributes: features: A17 Pro chip, 120\u202fHz ProMotion display, USB‑C fast charging; dimensions: compact\nMaterial: titanium frame, Ceramic Shield glass\nWarranty: 1‑year limited warranty\n'

In [12]:
%pip install python-dotenv openai



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY")
)



In [47]:
import time

results = []

for _, row in df.iterrows():

    product_prompt = format_product(row)

    start_time = time.time()

    response = client.chat.completions.create(
        model="meta-llama/Meta-Llama-3.1-8B-Instruct",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": product_prompt}
        ],
        temperature=0,
    
    )

    end_time = time.time()

    latency_ms = (end_time - start_time) * 1000

    results.append({
        "product_prompt": product_prompt,
        "description": response.choices[0].message.content,
        "latency_ms": latency_ms,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens
    })
    
metrics_df = pd.DataFrame(results)


In [48]:
metrics_df.head()

,product_prompt,description,latency_ms,input_tokens,output_tokens
0,\nProduct name: Apple iPhone 15 Pro\nAttribute...,"""Experience unparalleled performance with the ...",3205.854893,153,74
1,\nProduct name: Samsung Galaxy S24 Ultra\nAttr...,"""Unlock unparalleled smartphone performance wi...",3136.387348,156,82
2,\nProduct name: Google Pixel 8 Pro\nAttributes...,"""Unlock unparalleled photography and performan...",2109.575987,154,75
3,\nProduct name: Sony WH‑1000XM5 Headphones\nAt...,"""Immerse yourself in pure sound with the Sony ...",2385.568142,153,71
4,\nProduct name: Bose QuietComfort Ultra Earbud...,"""Immerse yourself in exceptional sound with th...",1948.470354,147,74


In [33]:
df = pd.concat([df, metrics_df], axis=1)

In [21]:
rubric_columns = [
    "fluency_score",
    "grammar_score",
    "tone_score",
    "length_score",
    "grounding_score",
    "latency_score",
    "cost_score"
]

for col in rubric_columns:
    df[col] = ""

In [25]:
df["final_score"] = ""


In [26]:
df.to_excel("assignment_01.xlsx", index=False)

# LLM Judge that grades product descriptions

In [ ]:
judge_prompt = """\
# Instruction
You are an expert evaluator. Your task is to evaluate the quality of the responses generated by an AI model.
You'll be provided with the input and an AI-generated responses.
You should first read the input carefully for analyzing the task, and then evaluate the quality of the responses based on the Criteria provided in the Evaluation section below.
You will assign the response a rating following the Rating Rubric and Evaluation Steps and only choose ratings from the Rating Rubric.

# Evaluation
## Metric Definition
You will be assessing product description generation quality.

## Criteria
Fluency: The response is well-organized and easy to read.

## Rating Rubric
3: (good). Sentences read naturally with smooth flow and no awkward phrasing.
2: (ok). Minor awkward phrasing but overall understandable and readable.
1: (bad). Multiple unnatural or difficult-to-follow sentences that disrupt readability.

## Criteria
Grammar: Correct spelling & punctuation.

## Rating Rubric
3: (good). No spelling or punctuation errors.
2: (ok). 1 minor grammar/spelling/punctuation error.
1: (bad). 2 or more grammar/spelling/punctuation errors.

## Criteria
Tone: Friendly, credible sales voice.

## Rating Rubric
3: (good). Consistently friendly, engaging, and credible sales tone throughout.
2: (ok). Mostly appropriate tone with minor inconsistency OR slightly neutral wording.
1: (bad). Tone inappropriate, overly formal, robotic, or not aligned with sales voice.

## Criteria
Length: 50–90 words

## Rating Rubric
3: (good). 50–90 words
2: (ok). 40–49 words OR 91–110 words
1: (bad). Less than 40 words OR more than 110 words

## Criteria
Grounding: The response contains exact information included only in the context.
## Rating Rubric
3: (good). All claims strictly supported by provided information.
2: (ok). One minor unsupported detail added.
1: (bad). Multiple unsupported claims OR contradicts provided information.

## Evaluation Steps
STEP 1: Assess the response in aspects of instruction following, fluency, grammar, tone, length and grounding according to the criteria.
STEP 2: Score based on the rubric.

# User Inputs and AI-generated Response

### Prompt
{product_prompt}
## AI-generated Response
{description}

For each criterion, please output:
Explanation: [brief explanation]
verdict: [good/ok/bad]
score: [3/2/1]


"""

In [50]:
evaluation_results = []

for _, row in metrics_df.iterrows():

    product_prompt = row["product_prompt"]
    description = row["description"]

    formatted_judge_prompt = judge_prompt.format(product_prompt=product_prompt, description=description)

    judge_response = client.chat.completions.create(
        model="Qwen/Qwen3-235B-A22B-Instruct-2507",
        messages=[
            {"role": "system", "content": formatted_judge_prompt}
        ],
        temperature=0
    )

    evaluation_results.append(
        judge_response.choices[0].message.content
    )

In [54]:
print(evaluation_results[0])

Explanation: The response is well-organized, with clear and natural sentence flow. Each feature is presented smoothly, and the transitions between ideas are logical and easy to follow.  
verdict: good  
score: 3  

Explanation: There are no spelling or punctuation errors. All sentences are correctly punctuated, and word usage is accurate.  
verdict: good  
score: 3  

Explanation: The tone is friendly, professional, and consistent with a credible sales voice. It highlights benefits in an engaging way without sounding robotic or overly formal.  
verdict: good  
score: 3  

Explanation: The response contains 86 words, which falls within the 50–90 word range.  
verdict: good  
score: 3  

Explanation: All information in the response is directly supported by the provided context—chip, display, charging, materials, warranty, and compactness is implied via "device" but not overstated. No external or unsupported claims are made.  
verdict: good  
score: 3
